<a href="https://colab.research.google.com/github/georgmandryka-cyber/beschwerden-antwortentwuerfe/blob/main/beschwerden_antwortentwuerfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install anthropic -q

from google.colab import userdata
import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def frage_claude(frage, system="Antworte auf Deutsch, kurz und praktisch."):
    antwort = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=2000,
        system=system,
        messages=[{"role": "user", "content": frage}]
    )
    teile = [block.text for block in antwort.content if block.type == "text"]
    return "".join(teile)

nachrichten = [
    "Das Produkt ist super, danke!",
    "Meine Bestellung ist nach zwei Wochen noch nicht da.",
    "Wie kann ich meine Adresse ändern?",
    "Na toll, schon wieder kaputt.",
    "Ich wollte nur sagen, dass ich umgezogen bin."
]
pd.DataFrame({"nachricht": nachrichten}).to_csv("eingang.csv", index=False)

df = pd.read_csv("eingang.csv")

# 1. Einordnen
df["kategorie"] = df["nachricht"].apply(
    lambda text: frage_claude(
        f"Ordne diese Nachricht genau einer Kategorie zu: Lob, Beschwerde, Frage oder Sonstiges. "
        f"Bei gemischten Nachrichten zählt das Hauptanliegen. "
        f"Antworte nur mit dem Wort. Nachricht: {text}"
    )
)

# 2. Antwortentwurf nur für Beschwerden
regeln = (
    "Du schreibst Antwortentwürfe für den Kundenservice. "
    "Antworte auf Deutsch, höflich und empathisch, in maximal 4 Sätzen. "
    "Schreibe durchgehend in der Wir-Form. "
    "Entschuldige dich kurz, nenne den nächsten Schritt und mache keine konkreten Versprechen "
    "(keine Fristen, keine Erstattungen, keine Wörter wie umgehend oder sofort). "
    "Beginne mit 'Guten Tag,' und ende immer mit 'Freundliche Grüße, Ihr Kundenservice'."
)

def entwurf(zeile):
    if zeile["kategorie"] == "Beschwerde":
        return frage_claude(f"Kundennachricht: {zeile['nachricht']}", system=regeln)
    return ""

df["antwortentwurf"] = df.apply(entwurf, axis=1)

df.to_csv("ausgang.csv", index=False)
pd.set_option("display.max_colwidth", 200)
print(df[["nachricht", "kategorie", "antwortentwurf"]])

for _, zeile in df[df["kategorie"] == "Beschwerde"].iterrows():
    print("Nachricht:", zeile["nachricht"])
    print(zeile["antwortentwurf"])
    print("-" * 40)


                                              nachricht   kategorie  \
0                         Das Produkt ist super, danke!         Lob   
1  Meine Bestellung ist nach zwei Wochen noch nicht da.  Beschwerde   
2                    Wie kann ich meine Adresse ändern?       Frage   
3                         Na toll, schon wieder kaputt.  Beschwerde   
4         Ich wollte nur sagen, dass ich umgezogen bin.   Sonstiges   

                                                                                                                                                                                            antwortentwurf  
0                                                                                                                                                                                                           
1  Guten Tag,\n\nes tut uns leid, dass Ihre Bestellung nach zwei Wochen noch nicht bei Ihnen angekommen ist, und wir können Ihren Ärger gut nachvollziehen. Damit w